📋 CHEATSHEET | Section 02 - POO

# CHEATSHEET - Programmation orientée objet

Référence rapide des cinq cours du dossier. Chaque bloc de code est exécutable.


## 1. Vocabulaire

| Terme | Définition | Code |
|---|---|---|
| Classe | plan, modèle | `class Compte:` |
| Objet, instance | exemplaire créé à partir du plan | `c = Compte("Ana")` |
| Attribut | donnée de l'objet (état) | `c.solde` |
| Méthode | fonction de l'objet (comportement) | `c.deposer(10)` |
| Constructeur | appelé à la création | `def __init__(self, ...)` |
| `self` | l'objet courant, dans la classe | `self.solde` |
| Attribut de classe | partagé par toutes les instances | `TAUX = 0.02` hors `__init__` |
| Encapsulation | cacher l'interne, exposer une interface | `_solde` + `@property` |
| Héritage | spécialiser une classe (« est un ») | `class Livret(Compte):` |
| Polymorphisme | même appel, comportement selon la classe | `for c in comptes: c.interets()` |
| Composition | un objet en contient un autre (« a un ») | `self.moteur = Moteur()` |
| Classe abstraite | contrat à implémenter | `class Forme(ABC)` + `@abstractmethod` |


## 2. Squelette de classe complet


In [ ]:
from functools import total_ordering

@total_ordering
class Compte:
    """Compte bancaire : solde toujours >= 0."""

    TAUX = 0.02                                   # attribut de classe

    def __init__(self, titulaire, solde=0.0):     # constructeur
        self.titulaire = titulaire                # public
        self.solde = solde                        # passe par le setter

    @property                                     # lecture : c.solde
    def solde(self):
        return self._solde

    @solde.setter                                 # écriture validée : c.solde = x
    def solde(self, valeur):
        if valeur < 0:
            raise ValueError("solde négatif")
        self._solde = valeur

    def deposer(self, montant):                   # méthode d'instance
        self.solde += montant

    @classmethod                                  # constructeur alternatif
    def depuis_texte(cls, texte):
        titulaire, solde = texte.split(":")
        return cls(titulaire, float(solde))

    @staticmethod                                 # fonction rangée dans la classe
    def valider_iban(iban):
        return iban.startswith("FR")

    def __repr__(self):                           # affichage technique
        return f"Compte({self.titulaire!r}, {self.solde})"

    def __eq__(self, autre):                      # == par l'état
        return isinstance(autre, Compte) and (self.titulaire, self.solde) == (autre.titulaire, autre.solde)

    def __hash__(self):                           # obligatoire avec __eq__ pour set/dict
        return hash((self.titulaire, self.solde))

    def __lt__(self, autre):                      # tri : sorted(), min(), max()
        return self.solde < autre.solde

c = Compte.depuis_texte("Ana:100")
c.deposer(50)
print(c, c == Compte("Ana", 150), sorted([c, Compte("Bo", 10)]))


## 3. Visibilité

| Python | UML | Sens |
|---|---|---|
| `nom` | `+` | public : interface, stable |
| `_nom` | `#` ou `-` | interne par convention, pas bloqué |
| `__nom` | `-` | renommé `_Classe__nom` (name mangling) |
| `@property` | `+` | attribut en lecture (et écriture si setter) |

Règle : commencer public ; passer à `_x` + `@property` quand il y a une règle (validation, invariant, calcul, lecture seule). Renvoyer des copies (`tuple(...)`) des collections internes.


## 4. Héritage, abstraction, composition


In [ ]:
from abc import ABC, abstractmethod

class Forme(ABC):                                 # contrat
    @abstractmethod
    def aire(self): ...

    def decrire(self):                            # concrète, partagée
        return f"{type(self).__name__} : {self.aire():.1f}"

class Rectangle(Forme):
    def __init__(self, l, h):
        self.l, self.h = l, h

    def aire(self):
        return self.l * self.h

class Carre(Rectangle):                           # héritage : un carré EST un rectangle
    def __init__(self, c):
        super().__init__(c, c)                    # réutilise le parent

    def decrire(self):
        return "carré, " + super().decrire()      # étend

class Dessin:                                     # composition : un dessin A des formes
    def __init__(self, formes):
        self.formes = list(formes)

    def aire_totale(self):
        return sum(f.aire() for f in self.formes) # polymorphisme

d = Dessin([Rectangle(2, 3), Carre(2)])
print([f.decrire() for f in d.formes], d.aire_totale())
print(isinstance(Carre(1), Forme), [k.__name__ for k in Carre.__mro__])


**Choisir** : « X est un Y » et X respecte tout le contrat de Y → héritage. « X a un Y », « X utilise Y », ou besoin de changer le comportement à l'exécution → composition. En cas de doute : composition.


## 5. Méthodes spéciales

| Syntaxe | Méthode | Syntaxe | Méthode |
|---|---|---|---|
| `print(a)`, `str(a)` | `__str__` (sinon `__repr__`) | `len(a)` | `__len__` |
| `repr(a)`, `[a]` | `__repr__` | `a[i]`, `for x in a` | `__getitem__` ou `__iter__` |
| `a == b` | `__eq__` (+ `__hash__`) | `x in a` | `__contains__` |
| `a < b` | `__lt__` (+ `total_ordering`) | `if a:` | `__bool__` (sinon `__len__`) |
| `a + b`, `a * n`, `n * a` | `__add__`, `__mul__`, `__rmul__` | `a(x)` | `__call__` |
| `with a:` | `__enter__`, `__exit__` | `a += b` | `__iadd__` (sinon `__add__`) |


## 6. Exceptions


In [ ]:
class ErreurMetier(Exception):                    # base du module
    pass

class SoldeInsuffisant(ErreurMetier):
    def __init__(self, manque):
        super().__init__(f"il manque {manque} €")
        self.manque = manque

def retirer(solde, montant):
    if montant <= 0:
        raise ValueError("montant invalide")      # mauvaise valeur
    if montant > solde:
        raise SoldeInsuffisant(montant - solde)   # erreur métier
    return solde - montant

try:
    retirer(10, 50)
except SoldeInsuffisant as e:                     # du plus précis...
    print("métier :", e, e.manque)
except (ValueError, TypeError) as e:              # ...au plus général
    print("argument :", e)
else:
    print("ok")                                   # si aucune exception
finally:
    print("toujours exécuté")

# Convertir une erreur technique en erreur métier sans perdre la cause
try:
    int("abc")
except ValueError as e:
    try:
        raise ErreurMetier("config invalide") from e
    except ErreurMetier as em:
        print(em, "<-", repr(em.__cause__))


Exceptions natives courantes : `ValueError`, `TypeError`, `KeyError`, `IndexError`, `AttributeError`, `ZeroDivisionError`, `FileNotFoundError`. Jamais `except:` nu ni `except Exception: pass`.


## 7. `@dataclass`


In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)                            # __init__, __repr__, __eq__, et < <= > >= avec order=True
class Produit:
    nom: str
    prix: float
    tags: list[str] = field(default_factory=list) # jamais [] directement

@dataclass(frozen=True)                           # immuable, hachable
class Coord:
    lat: float
    lon: float

print(Produit("stylo", 2.0), Produit("a", 1) < Produit("b", 1), {Coord(1, 2), Coord(1, 2)})


## 8. UML et Mermaid

| Relation | Phrase | Mermaid | Code |
|---|---|---|---|
| Héritage | est un | `Animal <\|-- Chat` | `class Chat(Animal)` |
| Réalisation | implémente | `Forme <\|.. Carre` | `class Carre(Forme)` avec `Forme(ABC)` |
| Composition | a un, créé avec lui | `Voiture *-- Moteur` | `self.moteur = Moteur()` |
| Agrégation | a des, qui préexistent | `Equipe o-- Joueur` | `self.joueurs = joueurs` |
| Association | connaît, utilise durablement | `Prof -- Cours` | attribut vers l'autre objet |
| Dépendance | utilise ponctuellement | `Facture ..> Tva` | paramètre ou appel |

Multiplicités : `1`, `0..1`, `*`, `1..*`, `2..4`, lues du côté opposé. Visibilité : `+` `-` `#`, `$` pour statique.

```mermaid
classDiagram
    Forme <|-- Rectangle
    Rectangle <|-- Carre
    Dessin o-- "0..*" Forme : contient
    class Forme {
        <<abstract>>
        +aire() float
        +decrire() str
    }
    class Dessin {
        +aire_totale() float
    }
```


## 9. Conception : principes et patrons

**Principes**

- **SRP** : une classe, une raison de changer. Alerte : « Gestionnaire », « Système », 500 lignes.
- **Composer plutôt qu'hériter** ; **programmer vers une interface** (ABC), pas une implémentation.
- **Substitution de Liskov** : une fille doit pouvoir remplacer sa mère partout.
- **KISS**, **DRY**, **YAGNI** : simple, sans doublon, sans « au cas où ».

**Patrons à reconnaître**

| Patron | Problème | Idée | Vu dans |
|---|---|---|---|
| Strategy | changer d'algorithme | une classe par algorithme, injectée par composition | 05, 09 |
| Factory | créer le bon objet selon une donnée | fonction ou `@classmethod` qui centralise la création | 01, 09 |
| Observer | prévenir des abonnés d'un changement | le sujet garde une liste d'observateurs et appelle leur `update()` | |
| Decorator | ajouter un comportement sans modifier la classe | un objet enveloppe un autre et délègue | |
| Adapter | faire parler deux interfaces incompatibles | une classe traduit les appels | |
| Singleton | une seule instance | à éviter : préférer passer l'objet en paramètre | |

**Checklist avant de coder** : noms → classes, données → attributs, verbes → méthodes, phrases de liaison → relations ; une classe sans comportement est peut-être un attribut ou une `dataclass` ; un patron seulement pour un problème réel.


## 10. Hors programme, pour aller plus loin

| Sujet | En une phrase | Quand ça devient utile |
|---|---|---|
| `__new__`, `__del__` | création bas niveau et finalisation d'un objet | jamais en pratique courante ; `__del__` n'est pas garanti |
| `__slots__` | fixe la liste des attributs, économise la mémoire | millions de petits objets |
| `typing.Protocol` | interface structurelle sans héritage (duck typing typé) | code typé avec mypy ou pyright |
| MRO, mixins | ordre de résolution en héritage multiple | frameworks (Django), rarement dans ton code |
| `inspect`, `getattr`, métaclasses | lire et modifier le code à l'exécution | outils, ORM, plugins |
| Gestionnaires de contexte | `with` grâce à `__enter__` / `__exit__` | fichiers, connexions, verrous |

Ressources : [Refactoring Guru](https://refactoring.guru/fr/design-patterns) (patrons illustrés, en français), [Python Patterns](https://python-patterns.guide), [mermaid.live](https://mermaid.live).
